# Chapter 17 — Background work: after, threads, queues, progress, and cancellation

**Book role:** Advanced GUI  
**Suggested book space:** about 10 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Use `after()` for scheduling short UI-thread tasks.
- Move blocking I/O or expensive calculations out of the UI event loop.
- Communicate worker results safely with queues.
- Design progress, cancellation, shutdown, and stale-result handling.

## 1. after is scheduling, not parallelism

`root.after(milliseconds, callback)` schedules a callback in the Tk event loop. It is useful for periodic refresh, blinking indicators, debounced search, incremental rendering, and polling a worker queue. It does not make the callback run concurrently. A callback that takes several seconds still blocks all other UI processing when it executes.

Never call `time.sleep()` on the UI thread to animate or wait for work. Schedule the next step with `after`. Store the returned job identifier if you may need to cancel it, and cancel outstanding scheduled callbacks before destroying the root. A repeating task should schedule its next occurrence only after its current work has completed, otherwise slow work can create overlapping jobs.

### Worked example 1: Incremental UI work with after

Each step is short, so the UI remains responsive. This is a simulated progression; real progress should be tied to completed work.

In [ ]:
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Scheduled progress")
root.geometry("420x170")
progress = ttk.Progressbar(root, maximum=100, length=340, mode="determinate")
progress.pack(padx=20, pady=(24, 8))
status = tk.StringVar(value="Ready")
ttk.Label(root, textvariable=status).pack()


def step(value=0):
    progress["value"] = value
    status.set(f"Progress: {value}%")
    if value < 100:
        root.after(40, step, value + 5)

root.after(200, step)
root.mainloop()

## 2. Threads and the Tk boundary

A worker thread can perform blocking file or database I/O and some pure-Python orchestration without blocking event handling. However, Tkinter is not generally thread-safe: worker threads should not directly update widgets. Instead, send a result or status message through `queue.Queue`, then poll the queue from the UI thread using `after()` and apply updates there.

For CPU-bound Python work, threads may not improve throughput due to the Global Interpreter Lock. Consider chunked processing, vectorized NumPy/pandas operations, a process pool, or a background service when appropriate. Start with measurement; do not add threading to every button because coordination, exception propagation, and cancellation require additional design.

### Worked example 2: Worker thread + queue + UI polling

The worker only puts messages into a queue; the main Tk thread changes widgets. This is a teaching example. Production code should keep a thread reference, use cooperative cancellation, catch and log failures, and shut down deliberately.

In [ ]:
import threading
import queue
import time
import tkinter as tk
from tkinter import ttk

root = tk.Tk()
root.title("Background task")
root.geometry("460x190")
messages = queue.Queue()
status = tk.StringVar(value="Ready")
button = ttk.Button(root, text="Start work")
button.pack(pady=16)
ttk.Label(root, textvariable=status).pack()


def worker():
    try:
        for step_number in range(1, 6):
            time.sleep(0.3)  # Simulate blocking work; never sleep in the UI callback.
            messages.put(("progress", step_number * 20))
        messages.put(("done", "Completed"))
    except Exception as exc:
        messages.put(("error", str(exc)))


def poll_queue():
    try:
        while True:
            kind, payload = messages.get_nowait()
            if kind == "progress":
                status.set(f"Progress: {payload}%")
            elif kind == "done":
                status.set(payload)
                button.configure(state="normal")
            elif kind == "error":
                status.set(f"Failed: {payload}")
                button.configure(state="normal")
    except queue.Empty:
        pass
    if root.winfo_exists():
        root.after(80, poll_queue)


def start():
    button.configure(state="disabled")
    status.set("Working…")
    threading.Thread(target=worker, daemon=True).start()

button.configure(command=start)
root.after(80, poll_queue)
root.mainloop()

## 3. Cancellation and stale results

Cancellation is a request, not a guarantee that a blocking driver call will stop immediately. A `threading.Event` can tell cooperative code to stop between steps; database drivers may have their own timeout and cancellation capabilities. On window close, signal cancellation, disable actions that must not repeat, and avoid destroying a view while a worker is trying to update it.

When users change filters while a query is running, the old result can arrive after a new one and overwrite it. Assign each request a monotonically increasing job ID; render a result only if it belongs to the current job. For long exports, communicate stage, rows processed, outcome, and errors. Progress bars should reflect measurable work rather than an arbitrary animation that reaches 100% before the operation finishes.

## Guided lab: Lab: responsive CSV import

Build a CSV import action that shows a loading state, disables duplicate import actions, reports progress if the format allows meaningful staged progress, and returns success/error through a queue. Never call widget methods from a worker. Add cancellation semantics or explain clearly why cancellation is unavailable during a blocking phase. Prevent old results from overwriting a newer request.

## Review questions and practice

1. Why does `after()` not make a slow callback parallel?
2. Why should worker threads not directly update Tk widgets?
3. What does a cancellation Event do and not guarantee?
4. How can an older query result overwrite a newer result?
5. When might a process pool be better than a thread?

## Further reading

- Tkinter threading model: https://docs.python.org/3/library/tkinter.html#threading-model
- Python queue: https://docs.python.org/3/library/queue.html
- Python threading: https://docs.python.org/3/library/threading.html

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.